Files originally from https://github.com/LucasSilvaFerreira/Perturb_Loader

In [11]:
import mudata as md
import anndata as ad
import perturbvi
import os
import pyro

In [3]:
def load_adata(data_dir = "."):
    rna_adata = ad.read_h5ad(f"{data_dir}/ann_exp.h5ad")
    rna_adata.obs['library_size']=rna_adata.X.sum(axis=1)
    rna_adata.varm['gene_tested'] = ad.read_h5ad(f"{data_dir}/ann_Element_x_tested_genes.h5ad").to_df().T
    grna_adata = ad.read_h5ad(f"{data_dir}/ann_guide.h5ad")
    grna_adata.varm['gene_targeted'] = ad.read_h5ad(f"{data_dir}/ann_Element_guide.h5ad").to_df().T
    mdata = md.MuData({'rna':rna_adata, 'grna':grna_adata})
    mdata.write_h5mu(f'{data_dir}/gasperini_pilot_highMOI.h5mu')
    return mdata

force = True
mudata_file = "gasperini_pilot_highMOI.h5mu"
data_dir = "../../../../Data/gasperini_pilot"
if mudata_file not in os.listdir(data_dir) or force:
    mdata = load_adata(data_dir)
else:
    mdata = md.read_h5mu(os.path.join(data_dir, mudata_file))

In [4]:
# subset down to only targeted genes
rna_subset = mdata['rna'][:,mdata['rna'].varm['gene_tested'].sum(axis=1).values > 0]
rna_subset =  mdata['rna'][:,['ACTB', 'AURKAIP1']]
mdata_subset = md.MuData({'rna':rna_subset, 'grna': mdata['grna']})
mdata_subset

MuData object with n_obs × n_vars = 47964 × 3117
  2 modalities
    rna:	47964 x 2
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count', 'library_size'
      varm:	'gene_tested'
    grna:	47964 x 3115
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count'
      varm:	'gene_targeted'

In [5]:
mdata['rna'].var_names[:100]

Index(['AL627309.1', 'AL627309.5', 'AP006222.1', 'AL732372.2', 'AL669831.3',
       'MTND1P23', 'MTND2P28', 'MTCO1P12', 'AC114498.2', 'MTATP6P1',
       'AL669831.1', 'LINC01409', 'LINC01128', 'LINC00115', 'AL645608.2',
       'NOC2L', 'KLHL17', 'PLEKHN1', 'HES4', 'ISG15', 'AGRN', 'AL390719.1',
       'RNF223', 'C1orf159', 'TTLL10', 'SDF4', 'B3GALT6', 'C1QTNF12', 'UBE2J2',
       'ACAP3', 'PUSL1', 'INTS11', 'CPTP', 'DVL1', 'MXRA8', 'AURKAIP1',
       'CCNL2', 'MRPL20-AS1', 'MRPL20', 'AL391244.2', 'VWA1', 'ATAD3C',
       'ATAD3B', 'ATAD3A', 'SSU72', 'AL645728.1', 'FNDC10', 'AL691432.2',
       'MIB2', 'MMP23B', 'CDK11B', 'FO704657.1', 'SLC35E2B', 'CDK11A',
       'AL031282.1', 'SLC35E2A', 'NADK', 'GNB1', 'TMEM52', 'CFAP74',
       'AL391845.2', 'GABRD', 'PRKCZ', 'AL590822.2', 'FAAP20', 'SKI', 'MORN1',
       'RER1', 'PEX10', 'PANK4', 'AL139246.5', 'TNFRSF14-AS1', 'TNFRSF14',
       'PRXL2B', 'AL512383.1', 'TPRG1L', 'WRAP73', 'TP73', 'TP73-AS1',
       'CCDC27', 'SMIM1', 'LRRC47', 'CEP1

In [8]:
perturbvi.PERTURBVI.setup_mudata(
    mdata_subset,
    size_factor_key="library_size",
    modalities={
        "rna_layer": 'rna',
        "perturbation_layer": 'grna',
    },
)

model = perturbvi.PERTURBVI(mdata_subset)
model.summary_stats

n_batch: 1
n_cells: 47964
n_extra_categorical_covs: 0
n_extra_continuous_covs: 0
n_perturbations: 3115
n_vars: 2

In [9]:
model.train(max_epochs=100, train_size=1, lr=0.1, batch_size=1024)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")


Epoch 100/100: 100%|██████████| 100/100 [01:06<00:00,  1.51it/s, v_num=1, elbo_train=4.81e+5]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████| 100/100 [01:06<00:00,  1.50it/s, v_num=1, elbo_train=4.81e+5]


In [ ]:
mdata['rna'].X.sum(axis=1).shape


(47964, 1)

In [14]:
for k, v in pyro.get_param_store().items():
    print (k, v.shape)

log_var_mean.mu torch.Size([2])
log_var_disp.mu torch.Size([2])
log_var_mean.sigma torch.Size([2])
log_var_disp.sigma torch.Size([2])
perturb_mean_lfc.mu torch.Size([3115, 2])
perturb_disp_lfc.mu torch.Size([3115, 2])
perturb_mean_lfc.scale_tril torch.Size([3115, 2, 2, 2])


In [15]:
perturb_mean_lfc.mu

NameError: name 'perturb_mean_lfc' is not defined